In [16]:
"""
Microsoft GraphRAG-Inspired Pipeline (Fixed Version)
=====================================================
Framework   : LangChain + NetworkX + OpenAI (using standard libraries)
LLM         : Azure OpenAI  (GPT-4o for extraction + generation)
Embeddings  : Azure OpenAI  (text-embedding-3-small, 1536-dim)
Search Modes: (1) Global Search  -- graph-based thematic analysis
              (2) Local Search   -- entity-focused retrieval
Data Source : Project Gutenberg public domain texts (plain .txt files)

This is a simplified, working version that doesn't depend on the
microsoft/graphrag package. It uses standard Python libraries:
- langchain for RAG components
- openai for Azure OpenAI integration
- networkx for graph operations
- chromadb for vector storage
- python-louvain for community detection
"""


"\nMicrosoft GraphRAG-Inspired Pipeline (Fixed Version)\n=====================================================\nFramework   : LangChain + NetworkX + OpenAI (using standard libraries)\nLLM         : Azure OpenAI  (GPT-4o for extraction + generation)\nEmbeddings  : Azure OpenAI  (text-embedding-3-small, 1536-dim)\nSearch Modes: (1) Global Search  -- graph-based thematic analysis\n              (2) Local Search   -- entity-focused retrieval\nData Source : Project Gutenberg public domain texts (plain .txt files)\n\nThis is a simplified, working version that doesn't depend on the\nmicrosoft/graphrag package. It uses standard Python libraries:\n- langchain for RAG components\n- openai for Azure OpenAI integration\n- networkx for graph operations\n- chromadb for vector storage\n- python-louvain for community detection\n"

In [1]:
# Download SpaCy language model
# Choose: en_core_web_sm (12MB, fast) or en_core_web_lg (780MB, better accuracy)
!python -m spacy download en_core_web_lg

     ---------------------------------------- 0.0/400.7 MB ? eta -:--:--
     ---------------------------------------- 0.8/400.7 MB 6.6 MB/s eta 0:01:01
     ---------------------------------------- 2.4/400.7 MB 6.4 MB/s eta 0:01:03
     ---------------------------------------- 4.2/400.7 MB 7.4 MB/s eta 0:00:54
      --------------------------------------- 6.3/400.7 MB 8.0 MB/s eta 0:00:50
      --------------------------------------- 8.4/400.7 MB 8.5 MB/s eta 0:00:47
     - ------------------------------------- 10.5/400.7 MB 8.8 MB/s eta 0:00:45
     - ------------------------------------- 12.8/400.7 MB 9.4 MB/s eta 0:00:42
     - ------------------------------------- 14.9/400.7 MB 9.7 MB/s eta 0:00:40
     - ------------------------------------- 17.3/400.7 MB 9.6 MB/s eta 0:00:41
     - ------------------------------------- 19.9/400.7 MB 9.9 MB/s eta 0:00:39
     -- ----------------------------------- 23.1/400.7 MB 10.3 MB/s eta 0:00:37
     -- ----------------------------------- 25.

In [ ]:

import os
import sys
import json
import asyncio
import logging
import urllib.request
import time
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple, Set
from collections import defaultdict, Counter

# Standard ML/AI libraries
try:
    import openai
    from openai import AzureOpenAI
except ImportError:
    print("ERROR: openai package not installed. Run: pip install openai")
    sys.exit(1)

try:
    import networkx as nx
    import community as community_louvain  # python-louvain
except ImportError:
    print("ERROR: networkx or python-louvain not installed. Run: pip install networkx python-louvain")
    sys.exit(1)

try:
    from langchain_core.documents import Document
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    print("ERROR: langchain not installed. Run: pip install langchain")
    sys.exit(1)

try:
    import chromadb
    from chromadb.config import Settings
except ImportError:
    print("ERROR: chromadb not installed. Run: pip install chromadb")
    sys.exit(1)

try:
    import tiktoken
except ImportError:
    print("ERROR: tiktoken not installed. Run: pip install tiktoken")
    sys.exit(1)  

# Hugging Face Embeddings
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("ERROR: sentence-transformers not installed. Run: pip install sentence-transformers")
    sys.exit(1)

# SpaCy for NLP-based entity extraction (ZERO LLM CALLS!)
try:
    import spacy
except ImportError:
    print("ERROR: spacy not installed. Run: pip install spacy")
    print("Also download a model: python -m spacy download en_core_web_lg")
    sys.exit(1)

C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0405 01:44:13.017000 31168 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("microsoft_graphrag")



In [5]:

class GraphRAGConfig:
    """Centralized configuration for the GraphRAG pipeline."""

    # --- Project Root Directory ---
    PROJECT_ROOT: str = "./graphrag_project"

    # --- Dataset: Plain-text files from Project Gutenberg ---
    TEXT_SOURCES: List[dict] = [
        {
            "name": "a_christmas_carol",
            "url":  "https://www.gutenberg.org/cache/epub/24022/pg24022.txt",
            "description": "A Christmas Carol by Charles Dickens (1843)",
        },
        {
            "name": "sherlock_holmes_adventures",
            "url":  "https://www.gutenberg.org/cache/epub/1661/pg1661.txt",
            "description": "The Adventures of Sherlock Holmes by Arthur Conan Doyle (1892)",
        },
    ]

    # --- Azure OpenAI Credentials (for LLM only) ---
    AZURE_OPENAI_API_KEY:         str = os.environ.get("AZURE_OPENAI_API_KEY", "")
    AZURE_OPENAI_ENDPOINT:        str = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
    AZURE_OPENAI_CHAT_DEPLOYMENT: str = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
    AZURE_OPENAI_API_VERSION:     str = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
    
    # --- Hugging Face Embeddings ---
    HUGGING_FACE_EMBEDDING_MODEL: str = "sentence-transformers/all-MiniLM-L6-v2"  # Fast, 384-dim
    # Alternative models:
    # "sentence-transformers/all-mpnet-base-v2"  # Better quality, 768-dim
    # "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # Multilingual

    # --- Entity Extraction Method ---
    # ⚠️ IMPORTANT: Choose extraction method
    USE_NLP_EXTRACTION: bool = True  # True = Local spaCy (FAST, FREE, NO LLM CALLS)
                                      # False = Azure OpenAI LLM (SLOW, COSTS $, N LLM CALLS)
    SPACY_MODEL: str = "en_core_web_lg"  # Only used if USE_NLP_EXTRACTION=True
    # Options: en_core_web_sm (12MB), en_core_web_md (40MB), en_core_web_lg (780MB), en_core_web_trf (440MB)

    # --- Indexing Parameters ---
    CHUNK_SIZE:           int = 1000     # characters per chunk
    CHUNK_OVERLAP:        int = 200      # character overlap
    MAX_ENTITIES_PER_CHUNK: int = 10     # max entities to extract per chunk
    ENTITY_TYPES: List[str] = [
        "person", "organization", "location", "event", "concept", "object"
    ]

    # --- Search Parameters ---
    TOP_K_CHUNKS:        int = 5
    TOP_K_ENTITIES:      int = 10
    MAX_RESPONSE_TOKENS: int = 2000
    TEMPERATURE:         float = 0.0


In [8]:
# ===========================================================================
# SECTION 2 -- AZURE OPENAI CLIENT WITH HUGGING FACE EMBEDDINGS
# ===========================================================================

class AzureOpenAIClient:
    """Wrapper for Azure OpenAI API calls (LLM) and Hugging Face  embeddings.
    
    Supports two modes for entity extraction:
    1. LLM-based: Uses Azure OpenAI (high quality, slow, expensive, N API calls)
    2. NLP-based: Uses spaCy NER (good quality, fast, free, local)
    """

    def __init__(self, config: GraphRAGConfig):
        self.config = config
        
        # Azure OpenAI for LLM/chat completions
        self.client = AzureOpenAI(
            api_key=config.AZURE_OPENAI_API_KEY,
            api_version=config.AZURE_OPENAI_API_VERSION,
            azure_endpoint=config.AZURE_OPENAI_ENDPOINT
        )
        self.tokenizer = tiktoken.get_encoding("cl100k_base")
        
        # Hugging Face for embeddings
        logger.info(f"Loading Hugging Face embedding model: {config.HUGGING_FACE_EMBEDDING_MODEL}")
        self.embedding_model = SentenceTransformer(config.HUGGING_FACE_EMBEDDING_MODEL)
        logger.info(f"Embedding dimension: {self.embedding_model.get_sentence_embedding_dimension()}")
        
        # Entity extraction mode
        if config.USE_NLP_EXTRACTION:
            logger.info(f"✓ Entity extraction mode: LOCAL NLP (spaCy {config.SPACY_MODEL})")
            logger.info(f"✓ LLM calls for entity extraction: ZERO (100% cost savings!)")
            try:
                self.nlp = spacy.load(config.SPACY_MODEL)
            except OSError:
                logger.error(f"SpaCy model '{config.SPACY_MODEL}' not found!")
                logger.error(f"Download it with: python -m spacy download {config.SPACY_MODEL}")
                raise
        else:
            logger.info(f"✓ Entity extraction mode: LLM (Azure OpenAI {config.AZURE_OPENAI_CHAT_DEPLOYMENT})")
            logger.info(f"⚠ Warning: This will make N LLM calls (one per chunk) - can be slow/expensive!")
            self.nlp = None

    def count_tokens(self, text: str) -> int:
        """Count tokens in text using tiktoken."""
        return len(self.tokenizer.encode(text))

    def get_embedding(self, text: str) -> List[float]:
        """Get embedding for text using Hugging Face Sentence Transformers."""
        try:
            # Encode text to get embedding
            embedding = self.embedding_model.encode(text, convert_to_numpy=True)
            return embedding.tolist()
        except Exception as e:
            logger.error(f"Hugging Face embedding error: {e}")
            # Return zero vector with correct dimension
            dim = self.embedding_model.get_sentence_embedding_dimension()
            return [0.0] * dim

    def chat_completion(self, messages: List[Dict[str, str]], max_tokens: int = 2000) -> str:
        """Get chat completion from Azure OpenAI."""
        try:
            response = self.client.chat.completions.create(
                model=self.config.AZURE_OPENAI_CHAT_DEPLOYMENT,
                messages=messages,
                max_tokens=max_tokens,
                temperature=self.config.TEMPERATURE
            )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Chat completion error: {e}")
            return f"Error: {str(e)}"

    def extract_entities_and_relationships(self, text: str) -> Dict[str, Any]:
        """Extract entities and relationships from text.
        
        Uses either:
        - Local NLP (spaCy) if config.USE_NLP_EXTRACTION = True (FAST, FREE)
        - Azure OpenAI LLM if config.USE_NLP_EXTRACTION = False (SLOW, COSTS $)
        """
        if self.config.USE_NLP_EXTRACTION:
            return self._extract_with_nlp(text)
        else:
            return self._extract_with_llm(text)
    
    def _extract_with_nlp(self, text: str) -> Dict[str, Any]:
        """Extract using spaCy NER (local, no LLM calls)."""
        doc = self.nlp(text)
        
        # Extract entities
        entities = []
        entity_set = set()
        max_entities = self.config.MAX_ENTITIES_PER_CHUNK
        
        for ent in doc.ents:
            if len(entities) >= max_entities:
                break
            if ent.text.strip() and len(ent.text) > 2:
                entity_type = self._map_spacy_entity_type(ent.label_)
                description = self._get_entity_context(ent, doc)
                
                entities.append({
                    "name": ent.text.strip(),
                    "type": entity_type,
                    "description": description
                })
                entity_set.add(ent.text.strip())
        
        # Extract relationships
        relationships = self._extract_nlp_relationships(doc, entity_set)
        
        return {"entities": entities, "relationships": relationships}
    
    def _map_spacy_entity_type(self, label: str) -> str:
        """Map spaCy entity labels to GraphRAG types."""
        mapping = {
            "PERSON": "person",
            "ORG": "organization",
            "GPE": "location",
            "LOC": "location",
            "FAC": "location",
            "EVENT": "event",
            "DATE": "event",
            "TIME": "event",
            "PRODUCT": "object",
            "WORK_OF_ART": "object",
            "LAW": "concept",
            "LANGUAGE": "concept",
            "NORP": "organization",
        }
        return mapping.get(label, "concept")
    
    def _get_entity_context(self, entity, doc, window: int = 5) -> str:
        """Get surrounding context for entity description."""
        sent = entity.sent
        start = max(0, entity.start - window)
        end = min(len(doc), entity.end + window)
        return doc[start:end].text.strip()[:150]
    
    def _extract_nlp_relationships(self, doc, entity_set: Set[str]) -> List[Dict[str, str]]:
        """Extract relationships from dependency parsing and co-occurrence."""
        relationships = []
        
        # Method 1: Verb-based relationships from dependency parsing
        for token in doc:
            if token.pos_ == "VERB":
                subjects = [child.text for child in token.children 
                           if child.dep_ in ("nsubj", "nsubjpass") and child.text in entity_set]
                objects = [child.text for child in token.children 
                          if child.dep_ in ("dobj", "pobj", "attr") and child.text in entity_set]
                
                for subj in subjects:
                    for obj in objects:
                        if subj != obj:
                            relationships.append({
                                "source": subj,
                                "target": obj,
                                "relation": token.lemma_,
                                "description": f"{subj} {token.text} {obj}"
                            })
        
        # Method 2: Co-occurrence relationships
        for sent in doc.sents:
            entities_in_sent = [ent.text for ent in sent.ents if ent.text in entity_set]
            if len(entities_in_sent) >= 2:
                for i, e1 in enumerate(entities_in_sent):
                    for e2 in entities_in_sent[i+1:]:
                        if not any(r['source'] == e1 and r['target'] == e2 for r in relationships):
                            relationships.append({
                                "source": e1,
                                "target": e2,
                                "relation": "co-occurs",
                                "description": f"Mentioned together"
                            })
        
        return relationships
    
    def _extract_with_llm(self, text: str) -> Dict[str, Any]:
        """Extract using Azure OpenAI LLM (original method)."""
        prompt = f"""Extract entities and relationships from the following text.

Text: {text}

Return a JSON object with this structure:
{{
    "entities": [
        {{"name": "Entity Name", "type": "person|organization|location|event|concept|object", "description": "brief description"}}
    ],
    "relationships": [
        {{"source": "Entity1", "target": "Entity2", "relation": "relationship type", "description": "brief description"}}
    ]
}}

Extract up to {self.config.MAX_ENTITIES_PER_CHUNK} most important entities."""

        messages = [
            {"role": "system", "content": "You are an expert at extracting structured information from text. Always return valid JSON."},
            {"role": "user", "content": prompt}
        ]

        response = self.chat_completion(messages, max_tokens=1000)
        
        # Try to parse JSON response
        try:
            # Extract JSON from markdown code blocks if present
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                response = response.split("```")[1].split("```")[0].strip()
            
            return json.loads(response)
        except json.JSONDecodeError:
            logger.warning(f"Failed to parse entity extraction response as JSON: {response[:100]}")
            return {"entities": [], "relationships": []}

In [9]:

# ===========================================================================
# SECTION 3 -- PROJECT INITIALIZATION
# ===========================================================================

def validate_credentials(config: GraphRAGConfig) -> None:
    """Validate required Azure OpenAI credentials."""
    missing = []
    if not config.AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")
    if not config.AZURE_OPENAI_ENDPOINT:
        missing.append("AZURE_OPENAI_ENDPOINT")
    
    if missing:
        raise EnvironmentError(
            f"Missing required environment variables: {missing}\n\n"
            "Set them before running:\n"
            "  export AZURE_OPENAI_API_KEY='your-azure-key'\n"
            "  export AZURE_OPENAI_ENDPOINT='https://your-resource.openai.azure.com/'\n"
            "  export AZURE_OPENAI_DEPLOYMENT='gpt-4o'\n"
            "  export AZURE_OPENAI_EMB_DEPLOYMENT='text-embedding-3-small'\n"
            "  export AZURE_OPENAI_API_VERSION='2024-02-15-preview'"
        )
    logger.info("Azure OpenAI credentials validated.")


def initialize_project(config: GraphRAGConfig) -> Path:
    """Create project directory structure."""
    root = Path(config.PROJECT_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    (root / "input").mkdir(exist_ok=True)
    (root / "output").mkdir(exist_ok=True)
    (root / "cache").mkdir(exist_ok=True)

    logger.info(f"GraphRAG project initialized at: {root.resolve()}")
    return root



In [10]:

# ===========================================================================
# SECTION 4 -- TEXT DATA ACQUISITION
# ===========================================================================

def download_texts(config: GraphRAGConfig, project_root: Path) -> List[Path]:
    """Download plain-text files from Project Gutenberg."""
    input_dir = project_root / "input"
    paths: List[Path] = []

    for source in config.TEXT_SOURCES:
        dest = input_dir / f"{source['name']}.txt"

        if dest.exists() and dest.stat().st_size > 1000:
            logger.info(f"Cached: {dest.name} ({dest.stat().st_size / 1024:.1f} KB)")
            paths.append(dest)
            continue

        logger.info(f"Downloading: {source['url']}")
        
        try:
            req = urllib.request.Request(
                source["url"],
                headers={"User-Agent": "Mozilla/5.0 (GraphRAG-Pipeline/1.0)"}
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw_bytes = resp.read()

            try:
                text = raw_bytes.decode("utf-8")
            except UnicodeDecodeError:
                text = raw_bytes.decode("latin-1")

            if len(text.strip()) < 500:
                raise RuntimeError(f"Downloaded text too short ({len(text)} chars)")

            dest.write_text(text, encoding="utf-8")
            logger.info(f"Saved: {dest.name} ({dest.stat().st_size / 1024:.1f} KB)")
            paths.append(dest)
            time.sleep(2.0)  # Be polite to Gutenberg servers

        except Exception as exc:
            logger.error(f"Failed to download {source['url']}: {exc}")
            raise

    return paths


def strip_gutenberg_headers(txt_path: Path) -> None:
    """Remove Project Gutenberg boilerplate from text file."""
    text = txt_path.read_text(encoding="utf-8")
    
    start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
    
    start_idx = text.find(start_marker)
    if start_idx != -1:
        start_idx = text.index("\n", start_idx) + 1
    else:
        start_idx = 0
    
    end_idx = text.find(end_marker)
    if end_idx == -1:
        end_idx = len(text)
    
    cleaned = text[start_idx:end_idx].strip()
    
    if len(cleaned) > 1000:
        txt_path.write_text(cleaned, encoding="utf-8")
        logger.info(f"Cleaned {txt_path.name}: {len(text)} → {len(cleaned)} chars")



In [11]:


# ===========================================================================
# SECTION 5 -- KNOWLEDGE GRAPH BUILDER
# ===========================================================================

class KnowledgeGraph:
    """Build and manage knowledge graph with entities and relationships."""

    def __init__(self, openai_client: AzureOpenAIClient):
        self.client = openai_client
        self.graph = nx.Graph()
        self.entity_descriptions: Dict[str, str] = {}
        self.entity_types: Dict[str, str] = {}
        self.entity_chunks: Dict[str, List[str]] = defaultdict(list)
        self.communities: Dict[int, List[str]] = {}
        self.community_summaries: Dict[int, str] = {}

    def add_entity(self, name: str, entity_type: str, description: str, chunk_id: str):
        """Add or update an entity in the graph."""
        name = name.strip()
        if not name:
            return
        
        if name not in self.graph:
            self.graph.add_node(name)
            self.entity_descriptions[name] = description
            self.entity_types[name] = entity_type
        else:
            # Merge descriptions - check if entity_descriptions exists for this name
            if name not in self.entity_descriptions:
                self.entity_descriptions[name] = description
                self.entity_types[name] = entity_type
            elif description and description not in self.entity_descriptions[name]:
                self.entity_descriptions[name] += f" {description}"
        
        self.entity_chunks[name].append(chunk_id)

    def add_relationship(self, source: str, target: str, relation: str):
        """Add a relationship between entities."""
        source = source.strip()
        target = target.strip()
        
        if not source or not target:
            return
        
        if source not in self.graph:
            self.graph.add_node(source)
        if target not in self.graph:
            self.graph.add_node(target)
        
        if self.graph.has_edge(source, target):
            # Increment weight if edge exists
            self.graph[source][target]['weight'] += 1
        else:
            self.graph.add_edge(source, target, relation=relation, weight=1)

    def detect_communities(self):
        """Detect communities using Louvain algorithm."""
        if len(self.graph) == 0:
            logger.warning("Graph is empty, cannot detect communities")
            return
        
        try:
            partition = community_louvain.best_partition(self.graph)
            
            # Group entities by community
            communities = defaultdict(list)
            for node, comm_id in partition.items():
                communities[comm_id].append(node)
            
            self.communities = dict(communities)
            logger.info(f"Detected {len(self.communities)} communities")
        except Exception as e:
            logger.error(f"Community detection failed: {e}")
            # Create single community as fallback
            self.communities = {0: list(self.graph.nodes())}

    def generate_community_summaries(self):
        """Generate LLM summaries for each community."""
        for comm_id, entities in self.communities.items():
            if not entities:
                continue
            
            # Get entity descriptions
            entity_info = []
            for entity in entities[:20]:  # Limit to top 20 entities
                desc = self.entity_descriptions.get(entity, "")
                entity_type = self.entity_types.get(entity, "unknown")
                entity_info.append(f"- {entity} ({entity_type}): {desc}")
            
            # Get relationships within community
            edges = []
            for source in entities:
                for target in entities:
                    if self.graph.has_edge(source, target):
                        relation = self.graph[source][target].get('relation', 'related to')
                        edges.append(f"- {source} {relation} {target}")
            
            # Generate summary
            prompt = f"""Summarize this community of related entities:

Entities:
{chr(10).join(entity_info[:50])}

Relationships:
{chr(10).join(edges[:30])}

Provide a 2-3 paragraph summary of what this community represents, its main themes, and significance."""

            messages = [
                {"role": "system", "content": "You are an expert at analyzing knowledge graphs and identifying thematic patterns."},
                {"role": "user", "content": prompt}
            ]
            
            summary = self.client.chat_completion(messages, max_tokens=500)
            self.community_summaries[comm_id] = summary
            logger.info(f"Generated summary for community {comm_id} ({len(entities)} entities)")

    def get_entity_community(self, entity: str) -> Optional[int]:
        """Get the community ID for an entity."""
        for comm_id, entities in self.communities.items():
            if entity in entities:
                return comm_id
        return None


In [12]:

# ===========================================================================
# SECTION 6 -- INDEXING PIPELINE
# ===========================================================================

class GraphRAGIndexer:
    """Build index with chunks, entities, relationships, and communities."""

    def __init__(self, config: GraphRAGConfig, openai_client: AzureOpenAIClient):
        self.config = config
        self.client = openai_client
        self.kg = KnowledgeGraph(openai_client)
        self.chunks: List[Document] = []
        self.chunk_embeddings: List[List[float]] = []
        self.vector_db = None

    def chunk_documents(self, doc_paths: List[Path]) -> List[Document]:
        """Split documents into chunks."""
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.config.CHUNK_SIZE,
            chunk_overlap=self.config.CHUNK_OVERLAP,
            length_function=len,
        )
        
        all_chunks = []
        for path in doc_paths:
            text = path.read_text(encoding="utf-8")
            chunks = text_splitter.create_documents(
                [text],
                metadatas=[{"source": path.name}]
            )
            all_chunks.extend(chunks)
        
        logger.info(f"Created {len(all_chunks)} text chunks")
        return all_chunks

    def extract_knowledge_graph(self, chunks: List[Document]):
        """Extract entities and relationships from chunks."""
        for idx, chunk in enumerate(chunks):
            chunk_id = f"chunk_{idx}"
            logger.info(f"Processing chunk {idx + 1}/{len(chunks)}")
            
            # Extract entities and relationships
            extraction = self.client.extract_entities_and_relationships(chunk.page_content)
            
            # Add entities to graph
            for entity in extraction.get("entities", []):
                self.kg.add_entity(
                    name=entity.get("name", ""),
                    entity_type=entity.get("type", "unknown"),
                    description=entity.get("description", ""),
                    chunk_id=chunk_id
                )
            
            # Add relationships to graph
            for rel in extraction.get("relationships", []):
                self.kg.add_relationship(
                    source=rel.get("source", ""),
                    target=rel.get("target", ""),
                    relation=rel.get("relation", "related to")
                )
            
            time.sleep(0.5)  # Rate limiting

    def create_embeddings(self, chunks: List[Document]) -> List[List[float]]:
        """Create embeddings for all chunks."""
        embeddings = []
        for idx, chunk in enumerate(chunks):
            if idx % 10 == 0:
                logger.info(f"Embedding chunk {idx + 1}/{len(chunks)}")
            embedding = self.client.get_embedding(chunk.page_content)
            embeddings.append(embedding)
            time.sleep(0.2)  # Rate limiting
        
        return embeddings

    def build_vector_store(self, project_root: Path):
        """Build ChromaDB vector store."""
        chroma_path = str(project_root / "output" / "chroma_db")
        
        # Initialize ChromaDB
        chroma_client = chromadb.PersistentClient(path=chroma_path)
        collection = chroma_client.get_or_create_collection(name="chunks")
        
        # Add chunks to vector store
        for idx, (chunk, embedding) in enumerate(zip(self.chunks, self.chunk_embeddings)):
            collection.add(
                ids=[f"chunk_{idx}"],
                embeddings=[embedding],
                documents=[chunk.page_content],
                metadatas=[chunk.metadata]
            )
        
        self.vector_db = collection
        logger.info(f"Built vector store with {len(self.chunks)} chunks")

    def run_indexing(self, doc_paths: List[Path], project_root: Path):
        """Run complete indexing pipeline."""
        logger.info("=== Starting Indexing Pipeline ===")
        
        # Step 1: Chunk documents
        self.chunks = self.chunk_documents(doc_paths)
        
        # Step 2: Extract knowledge graph
        logger.info("Extracting entities and relationships...")
        self.extract_knowledge_graph(self.chunks)
        
        # Step 3: Detect communities
        logger.info("Detecting communities...")
        self.kg.detect_communities()
        
        # Step 4: Generate community summaries
        logger.info("Generating community summaries...")
        self.kg.generate_community_summaries()
        
        # Step 5: Create embeddings
        logger.info("Creating embeddings...")
        self.chunk_embeddings = self.create_embeddings(self.chunks)
        
        # Step 6: Build vector store
        logger.info("Building vector store...")
        self.build_vector_store(project_root)
        
        # Save graph
        graph_path = project_root / "output" / "knowledge_graph.gexf"
        nx.write_gexf(self.kg.graph, graph_path)
        logger.info(f"Saved knowledge graph to {graph_path}")
        
        logger.info("=== Indexing Complete ===")



In [13]:

# ===========================================================================
# SECTION 7 -- QUERY ENGINE
# ===========================================================================

class GraphRAGQuery:
    """Query engine for global and local search."""

    def __init__(self, config: GraphRAGConfig, indexer: GraphRAGIndexer):
        self.config = config
        self.indexer = indexer
        self.client = indexer.client

    def global_search(self, query: str) -> str:
        """Global search using community summaries."""
        logger.info(f"Running Global Search: {query}")
        
        # Get all community summaries
        summaries_text = []
        for comm_id, summary in self.indexer.kg.community_summaries.items():
            entities = self.indexer.kg.communities.get(comm_id, [])
            summaries_text.append(f"Community {comm_id} ({len(entities)} entities):\n{summary}\n")
        
        if not summaries_text:
            return "No communities found in the knowledge graph."
        
        # Query LLM with community summaries
        context = "\n---\n".join(summaries_text)
        prompt = f"""Based on the following community summaries from a knowledge graph, answer the question.

Community Summaries:
{context}

Question: {query}

Provide a comprehensive answer based on the thematic patterns across all communities."""

        messages = [
            {"role": "system", "content": "You are an expert at synthesizing information from knowledge graphs to answer thematic questions."},
            {"role": "user", "content": prompt}
        ]
        
        answer = self.client.chat_completion(messages, max_tokens=self.config.MAX_RESPONSE_TOKENS)
        return answer

    def local_search(self, query: str) -> str:
        """Local search using entity graph and vector similarity."""
        logger.info(f"Running Local Search: {query}")
        
        # Get query embedding
        query_embedding = self.client.get_embedding(query)
        
        # Search vector store
        if self.indexer.vector_db is None:
            return "Vector store not initialized."
        
        results = self.indexer.vector_db.query(
            query_embeddings=[query_embedding],
            n_results=self.config.TOP_K_CHUNKS
        )
        
        # Get relevant chunks
        chunks_text = []
        if results and results['documents']:
            chunks_text = results['documents'][0]
        
        # Extract mentioned entities from query
        words = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b', query)
        relevant_entities = [w for w in words if w in self.indexer.kg.graph]
        
        # Get entity information
        entity_info = []
        for entity in relevant_entities:
            desc = self.indexer.kg.entity_descriptions.get(entity, "")
            entity_type = self.indexer.kg.entity_types.get(entity, "unknown")
            entity_info.append(f"- {entity} ({entity_type}): {desc}")
            
            # Get neighbors
            neighbors = list(self.indexer.kg.graph.neighbors(entity))[:5]
            for neighbor in neighbors:
                if self.indexer.kg.graph.has_edge(entity, neighbor):
                    relation = self.indexer.kg.graph[entity][neighbor].get('relation', 'related to')
                    entity_info.append(f"  → {entity} {relation} {neighbor}")
        
        # Build context
        context_parts = []
        if chunks_text:
            context_parts.append("Relevant Text Chunks:\n" + "\n---\n".join(chunks_text))
        if entity_info:
            context_parts.append("Entity Information:\n" + "\n".join(entity_info))
        
        context = "\n\n".join(context_parts)
        
        if not context:
            return "No relevant information found."
        
        # Query LLM
        prompt = f"""Based on the following context, answer the question.

Context:
{context}

Question: {query}

Provide a detailed answer based on the available information."""

        messages = [
            {"role": "system", "content": "You are an expert at answering questions using both structured knowledge graphs and text passages."},
            {"role": "user", "content": prompt}
        ]
        
        answer = self.client.chat_completion(messages, max_tokens=self.config.MAX_RESPONSE_TOKENS)
        return answer



In [14]:

# ===========================================================================
# SECTION 8 -- MAIN ORCHESTRATOR
# ===========================================================================

def main():
    """Main execution pipeline."""
    print("=" * 80)
    print("GraphRAG Pipeline (Fixed Version)")
    print("=" * 80)
    
    config = GraphRAGConfig()
    
    # Validate credentials
    try:
        validate_credentials(config)
    except EnvironmentError as e:
        logger.error(str(e))
        return
    
    # Initialize project
    project_root = initialize_project(config)
    
    # Download and clean texts
    logger.info("Downloading source texts...")
    txt_paths = download_texts(config, project_root)
    for txt_path in txt_paths:
        strip_gutenberg_headers(txt_path)
    
    # Initialize Azure OpenAI client
    openai_client = AzureOpenAIClient(config)
    
    # Run indexing
    indexer = GraphRAGIndexer(config, openai_client)
    indexer.run_indexing(txt_paths, project_root)
    
    # Initialize query engine
    query_engine = GraphRAGQuery(config, indexer)
    
    # Demo queries
    logger.info("\n" + "=" * 80)
    logger.info("Running Demo Queries")
    logger.info("=" * 80)
    
    # Global search queries
    global_queries = [
        "What are the major themes across both stories?",
        "How do the main characters undergo transformation?",
    ]
    
    for q in global_queries:
        print(f"\n{'=' * 80}\nGLOBAL SEARCH\nQuery: {q}\n{'=' * 80}")
        answer = query_engine.global_search(q)
        print(f"\nAnswer:\n{answer}\n")
    
    # Local search queries
    local_queries = [
        "Who is Ebenezer Scrooge and what are his key relationships?",
        "What is Sherlock Holmes' methodology for solving crimes?",
    ]
    
    for q in local_queries:
        print(f"\n{'=' * 80}\nLOCAL SEARCH\nQuery: {q}\n{'=' * 80}")
        answer = query_engine.local_search(q)
        print(f"\nAnswer:\n{answer}\n")
    
    logger.info("\n" + "=" * 80)
    logger.info("Pipeline Complete!")
    logger.info("=" * 80)



In [15]:
main()

GraphRAG Pipeline (Fixed Version)
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Azure OpenAI credentials validated.
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | GraphRAG project initialized at: C:\Users\dhanu\Downloads\RAG_types\02_chunking_document_rag\graphrag_project
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Downloading source texts...
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Cached: a_christmas_carol.txt (172.3 KB)
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Cached: sherlock_holmes_adventures.txt (597.2 KB)
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Cleaned a_christmas_carol.txt: 169230 → 169230 chars
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Cleaned sherlock_holmes_adventures.txt: 574123 → 574123 chars
2026-04-05 01:46:10 | INFO     | microsoft_graphrag | Loading Hugging Face embedding model: sentence-transformers/all-MiniLM-L6-v2
2026-04-05 01:46:10 | INFO     | sentence_transformers.SentenceTransformer | 

2026-04-05 01:46:11 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-05 01:46:11 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/README.md "HTTP/1.1 200 OK"
2026-04-05 01:46:12 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 01:46:12 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-05 01:46:12 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/sentence_bert_config.json "HTTP/1.1 307 Tempora

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4192.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-05 01:46:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 01:46:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-04-05 01:46:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-05 01:46:13 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-05 01:46:14 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=fals

KeyboardInterrupt: 